# 03. 기초 통계
**목표**: 요약 통계, IQR 이상치 탐지, 상관계수를 이해하고 직접 구현합니다. (예상 4~5시간)

코드보다 **해석** 이 중요합니다. 미션은 모든 해석에 *수치 근거 1개 이상 + 시사점 1문장 이상* 을 요구합니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
price = pd.Series(rng.normal(50000, 10000, 300).round(-2))
price.iloc[:5] = [250000, 300000, 5000, 280000, 1000]   # 일부러 이상치 삽입
price.describe()

## 1. 평균, 중앙값, 표준편차
- **평균**: 모든 값의 합 ÷ 개수. 극단값에 **민감**합니다.
- **중앙값**: 크기순 정렬 시 가운데 값. 극단값에 **강합니다**.
- **표준편차**: 값이 평균에서 평균적으로 얼마나 떨어져 있는지(퍼진 정도).

In [ ]:
print("평균  :", round(price.mean()))
print("중앙값:", round(price.median()))
print("표준편차:", round(price.std()))

clean = price.iloc[5:]
print("--- 이상치 5개를 뺀 경우 ---")
print("평균  :", round(clean.mean()))
print("중앙값:", round(clean.median()))

> 이상치 몇 개만으로 평균은 크게 움직이지만 중앙값은 거의 그대로입니다. 분포가 치우친 데이터에서는 **평균과 중앙값을 같이 보는 것**이 안전합니다.

## 2. 사분위수와 IQR
- **Q1**: 하위 25% 지점, **Q2**: 중앙값, **Q3**: 상위 25% 지점
- **IQR = Q3 − Q1**: 가운데 50% 데이터가 퍼진 폭

In [ ]:
q1 = price.quantile(0.25)
q2 = price.quantile(0.50)
q3 = price.quantile(0.75)
iqr = q3 - q1
print(q1, q2, q3, iqr)

## 3. IQR 이상치 탐지 직접 구현 ⭐ (미션 4.4)

**규칙**
- 하한 = Q1 − 1.5 × IQR
- 상한 = Q3 + 1.5 × IQR
- 하한보다 작거나 상한보다 크면 이상치

1.5는 관례적인 값이며 `threshold` 인자로 바꿀 수 있게 만듭니다. (미션 4.2: 파라미터는 인자로)

In [ ]:
def iqr_bounds(s: pd.Series, threshold: float = 1.5):
    """IQR 방식의 (하한, 상한)을 반환"""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - threshold * iqr, q3 + threshold * iqr


def detect_outliers_iqr(s: pd.Series, threshold: float = 1.5) -> pd.Series:
    """이상치 여부를 불리언 Series로 반환"""
    lower, upper = iqr_bounds(s, threshold)
    return (s < lower) | (s > upper)


lower, upper = iqr_bounds(price)
mask = detect_outliers_iqr(price)
print(f"정상 범위: {lower:,.0f} ~ {upper:,.0f}")
print("이상치 개수:", mask.sum())
print(price[mask].to_list())

threshold 를 키우면 더 극단적인 값만 이상치로 잡습니다.

In [ ]:
for t in [1.0, 1.5, 3.0]:
    print(f"threshold={t}: 이상치 {detect_outliers_iqr(price, t).sum()}개")

### 이상치를 어떻게 처리할까?
| 방법 | 설명 | 주의 |
|---|---|---|
| 제거 | 이상치 행 삭제 | 데이터 손실 |
| 클리핑(윈저라이징) | 상·하한 값으로 대체 | 행 수 유지 |
| 유지 | 진짜 의미 있는 값이면 보존 | VIP 고객의 큰 구매는 이상치가 아니라 **핵심 정보**일 수 있음 |

In [ ]:
price_removed = price[~mask]
price_clipped = price.clip(lower, upper)
print("원본   :", len(price), round(price.mean()), round(price.std()))
print("제거   :", len(price_removed), round(price_removed.mean()), round(price_removed.std()))
print("클리핑 :", len(price_clipped), round(price_clipped.mean()), round(price_clipped.std()))

## 4. 상관계수
두 변수가 **함께 움직이는 정도**를 -1 ~ +1 로 나타냅니다.

| 값 | 의미 |
|---|---|
| +1 에 가까움 | 한쪽이 커질 때 다른 쪽도 커짐 |
| 0 근처 | 선형 관계 거의 없음 |
| -1 에 가까움 | 한쪽이 커질 때 다른 쪽은 작아짐 |

대략 |r| < 0.3 약함, 0.3~0.7 보통, > 0.7 강함 으로 봅니다.

In [ ]:
n = 500
qty = rng.integers(1, 10, n)
unit_price = rng.normal(30000, 8000, n)
amount = qty * unit_price + rng.normal(0, 5000, n)   # qty와 amount는 강한 관계
noise = rng.normal(0, 1, n)                          # 아무 관계 없음

d = pd.DataFrame({"qty": qty, "unit_price": unit_price, "amount": amount, "noise": noise})
d.corr().round(2)

In [ ]:
print("qty ~ amount  :", round(d["qty"].corr(d["amount"]), 2))
print("qty ~ noise   :", round(d["qty"].corr(d["noise"]), 2))

### ⚠️ 상관 ≠ 인과
"아이스크림 판매량과 익사 사고가 상관이 높다"고 아이스크림이 사고의 원인은 아닙니다 (여름이라는 제3의 요인).
해석에는 "관련이 있다" 까지만 말하고, 원인은 단정하지 않습니다.

## 5. 좋은 해석 문장 만들기
**나쁜 예**: "price의 평균은 52,000이다."  (숫자만 있고 의미가 없음)

**좋은 예**: "price 평균은 52,000원, 중앙값은 50,100원으로 비슷해 분포가 대체로 대칭이다. 따라서 평균을 대표값으로 써도 무방하다."
→ 수치 근거(평균, 중앙값) + 시사점(평균을 대표값으로 사용 가능)

### ✏️ 직접 해보기
위 `price` 에서 `threshold=3.0` 일 때의 상한을 구하고, 그 상한을 넘는 값이 몇 개인지 구하세요.

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [ ]:
lo, up = iqr_bounds(price, 3.0)
print(round(up), (price > up).sum())

### ✏️ 직접 해보기
`d` 에서 `unit_price` 와 `amount` 의 상관계수를 구하고, 수치 근거 + 시사점 형식으로 한 문장씩 해석을 써 보세요.

In [ ]:
# 여기에 직접 작성해 보세요

✅ 다음: `04_visualization`